In [ ]:
import functools as ft
import itertools

import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from jaxtyping import Array, Float, Integer, Scalar, ScalarLike
from mdpax.core.problem import Problem
from mdpax.utils.spaces import create_range_space
from mdpax.utils.types import (
    ActionSpace,
    ActionVector,
    Policy,
    RandomEventSpace,
    RandomEventVector,
    Reward,
    StateSpace,
    StateVector,
)

import fairsim
from fairsim.model.joint import (
    joint_opt_demographic_parity,
    joint_opt_equal_opportunity,
    ternary_maximize_1d,
)
from fairsim.model.mdp import (
    transition,
    transition_kernel_construct,
)
from fairsim.model.myopic import (
    opt_choice_prob,
    opt_tpr,
    opt_unconstrained,
)
from fairsim.util import KeyGen, tree_stack, tree_unstack

sns.set_theme(context='notebook', style='whitegrid')

In [ ]:
n_scores = 17
succ_prob = jnp.linspace(0.1, 0.9, n_scores)
distrib_x = (1 + jnp.arange(n_scores)) 
distrib_x = distrib_x / distrib_x.sum()
reward = succ_prob - 0.7

kernel = transition_kernel_construct(
    repay=jnp.array([0, 1, 1]),
    default=jnp.array([1, 0, 0]),
    reject=jnp.array([0.1, 0.8, 0.1]),
    noise=jnp.array([1.]),
)
weights = jnp.array([0.8, 0.2])

init_distribs = (distrib_x, distrib_x[::-1])
n_iter = 10000
def sim_step(distribs, _, mode):
    if mode == 'dp':
        th, policies = joint_opt_demographic_parity(tree_stack(distribs), weights, reward)
    elif mode == 'eo':
        th, policies = joint_opt_equal_opportunity(tree_stack(distribs), weights, succ_prob, reward)
    else:
        th, policies = None, tuple(opt_unconstrained(distrib, reward) for distrib in distribs)
    distribs_t = tuple(
        transition(
            distrib, policy, succ_prob, kernel
        )
        for distrib, policy in zip(distribs, policies)
    )
    return distribs_t, (distribs, th)
def dist(distribs):
    cdfs = jnp.array([distrib.cumsum() for distrib in distribs])
    return cdfs.ptp(axis=0).max()

result = {}
for mode in ['dp', 'eo', 'unconstrained']:
    distribs, (history, ths) = jax.lax.scan(ft.partial(sim_step, mode=mode), init_distribs, None, length=n_iter + 1)
    dists = jax.vmap(dist)(history)
    result[mode] = {
        "history": history,
        "thresholds": ths,
        "distances": dists,
    }

In [ ]:
fig, ax = plt.subplots()
linestyles = {'dp': '-', 'eo': (0, (4, 4)), 'unconstrained': '--'}
names = {'dp': 'DemoPar', 'eo': 'EqOpp', 'unconstrained': 'Unconstrained'}
for mode in result:
    dists = result[mode]["distances"]
    sns.lineplot(x=jnp.arange(len(dists)), y=dists, ax=ax, label=names[mode], linestyle=linestyles[mode])
ax.set_yscale('log')
ax.set_title("Distance between distributions over time")

In [ ]:
fig, ax = plt.subplots()
for mode in result:
    ths = result[mode]["thresholds"]
    if ths is not None:
        sns.lineplot(x=jnp.arange(len(ths)), y=ths, ax=ax, label=names[mode])
# ax.set_yscale('log')
ax.set_title("Thresholds over time")

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(16, 8), layout='constrained', sharex=True)
sampled_times = [0, 3, 10, 100, 1000, 10000]
ht = {mode: tree_unstack(result[mode]["history"]) for mode in result.keys()}
for ax, (mode, t) in zip(axes.flatten(), itertools.product(result.keys(), sampled_times)):
    for distrib in ht[mode][t]:
        sns.barplot(x=jnp.arange(n_scores), y=distrib, ax=ax, alpha=0.5)
        # sns.lineplot(x=jnp.arange(n_scores), y=distrib.cumsum(), ax=ax)
    ax.set_title(f"{names[mode]}, t={t}")
# axes[0, -1].legend()
plt.show()